In [2]:
import sys
import json
from pathlib import Path

sys.path.append(str(Path().resolve().parents[0]))
sys.path.append(str(Path().resolve().parents[1]))


from main.persona_types.Persona import *
from main.transcript_types.Transcript import *
from main.calendar_types.Calendar import *

In [ ]:
import os
os.environ["OPENAI_API_KEY"] = "key"

In [4]:
conversation_context_prompt = """
You are generating conversation scene ideas, not full transcripts.

Focal persona:
{persona_summary}

Close circle personas:
{close_personas_summary}

Generate exactly 20 scenes describing plausible message exchanges between the focal persona and members of their close circle.

Requirements:
- Each scene must involve the focal persona as the speaker.
- Each scene must involve exactly one recipient chosen from the close circle personas.
- Base every scene only on the persona information provided above.
- Make the scenes realistic, ordinary, and socially plausible.
- Keep each summary concise, but specific enough to convey the topic, tone, and social context of the exchange.
- Vary the scenes across topics, tone, and relationship dynamics.
- The exchanges can include casual check-ins, practical updates, planning, debates, encouragement, conflict, jokes, or personal discussion.
- Do not write the full messages or dialogue.
- Do not invent people outside the provided close circle.
- Do not repeat the same kind of scene with only minor wording changes.

Output a structured list of Scene objects.
"""

In [5]:
import json
with open("../expand_and_socialize/expanded_personas/personas.json") as f:
    personas = json.load(f)

# print(personas[0])
focal_persona = personas[0]
close_circle = personas[1:]
print(focal_persona)

{'persona_id': 'e7c0574639a244c8972c92aab9501035', 'demographics': {'name': 'Mary Alberti', 'age': 28, 'gender': 'Female', 'education': 'High school', 'occupation': 'Fast food or counter worker', 'income_bracket': 'Under $35k', 'location': 'Madison, WI 53717, USA', 'marital_status': 'Never married'}, 'psych_traits': {'openness': 0.61, 'conscientiousness': 0.9, 'extraversion': 0.56, 'agreeableness': 0.74, 'neuroticism': 0.42}, 'preferences_and_interests': {'health_and_wellness': 'Mary considers her health generally good and focuses on consistency rather than extremes. She runs a few times a week, often around Lake Mendota, and uses exercise as both fitness and stress relief. She values sleep, meal prep, stretching, decent shoes, and routines to manage fatigue, headaches, foot soreness, and lower back strain from long shifts. She is mentally steadier when life feels organized, and she uses budgeting, journaling, and weekly planning as part of her wellness routine.', 'food': 'She works in

In [6]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-5.4", temperature=0, seed=42)
scene_llm = llm.with_structured_output(SceneList)
transcript_llm = llm.with_structured_output(Transcript)
calendar_llm = llm.with_structured_output(CalendarLog)

In [7]:
from typing_extensions import TypedDict

class State(TypedDict):
    persona: Persona
    close_circle: list[Persona]
    scenes: SceneList | None
    transcripts: list[Transcript] | None
    calendar_log: CalendarLog | None


In [8]:
from langchain.messages import HumanMessage, SystemMessage

def generate_scenes(state: State) -> dict:
    persona = state["persona"]
    close_personas_circle = state["close_circle"]

    scenes: SceneList = scene_llm.invoke([
        SystemMessage(
            content=(
                "You generate realistic conversation scene ideas, not full transcripts. "
                "Generate exactly 20 Scene objects. "
                "Each scene must involve the focal persona as the speaker and exactly one recipient "
                "from the provided close circle. "
                "Base the scenes only on the provided persona summaries. "
                "Make them realistic, ordinary, socially plausible, concise, and diverse. "
                "Do not invent people outside the provided close circle. "
                "Do not write full dialogue."
            )
        ),
        HumanMessage(content="Focal persona summary:\n" + json.dumps(persona, indent=2)),
        HumanMessage(content="Close circle personas summary:\n" + json.dumps(close_personas_circle, indent=2)),
    ])

    return {
        "scenes": scenes.scenes
    }


def generate_message_logs(state: State):
    persona = state["persona"]
    close_personas_circle = state["close_circle"]
    scenes = state["scenes"]

    transcript_list = []

    for close_persona in close_personas_circle:
        res = transcript_llm.invoke([
            SystemMessage(
                content=(
                    "Generate a Transcript of messages between the given focal persona and the particular target persona in their close circle."
                    "You may refer to the list of scenes for direct usage or inspiration."
                    "Use the semantic qualities of the given personas to adopt their tone and attitude in the conversation."
                    "The conversation should be anywhere from 10-50 turns representing the last two or three months of messenger (text, ig, whatsapp, etc) converstaion between these personas."
                )
            ),
            HumanMessage(
                content="Focal Persona:\n" + json.dumps(persona, indent=2)
            ),
            HumanMessage(
                content="Close Circle Personas:\n" + json.dumps(close_persona, indent=2)
            ),
            HumanMessage(
                content="Scene (conversation) list:\n" + json.dumps(
                    [scene.model_dump() for scene in scenes],
                    indent=2
                    )
            )
        ])
        transcript_list.append(res)

    return {"transcripts": transcript_list}


def generate_calendar_logs(state: State):
    persona = state["persona"]
    social_circle = state["close_circle"]

    res: CalendarLog = calendar_llm.invoke([
        SystemMessage(content=(
            "Create a CalendarLog for the given persona for the last 2 months."
            "Try to make the log realistic given the personas properties including preferences, interests, and close circle."
            "Do not list routine tasks (e.g. 'breakfast')."
        )),
        HumanMessage(content="Focal Persona: " + json.dumps(persona, indent=2)),
        HumanMessage(content="Close Circle Personas: " + json.dumps(social_circle, indent=2))
    ])

    return {"calendar_log": res}


In [9]:
from langgraph.graph import StateGraph, START, END

agent_builder = StateGraph(State)

agent_builder.add_node("generate_scenes", generate_scenes)
agent_builder.add_node("generate_transcripts", generate_message_logs)
agent_builder.add_node("generate_calendar_logs", generate_calendar_logs)

agent_builder.add_edge(START, "generate_scenes")
agent_builder.add_edge("generate_scenes", "generate_transcripts")
agent_builder.add_edge("generate_transcripts", "generate_calendar_logs")
agent_builder.add_edge("generate_calendar_logs", END)

agent = agent_builder.compile()

In [10]:
import os
os.makedirs("app_logs", exist_ok=True)

In [11]:
initial_state = {
    "persona": focal_persona,
    "close_circle": close_circle,
    "scenes": None,
    "transcripts": None,
    "calendar_log": None
}

result = agent.invoke(initial_state)

In [12]:
with open("app_logs/scenes.json", "w") as f:
    json.dump(
        [scene.model_dump() for scene in result["scenes"]],
        f,
        indent=2,
        default=str
    )

with open("app_logs/transcripts.json", "w") as f:
    json.dump(
        [t.model_dump() for t in result["transcripts"]],
        f,
        indent=2,
        default=str  # REQUIRED for datetime
    )

with open("app_logs/calendar_logs.json", "w") as f:
    json.dump(
        result["calendar_log"].model_dump(),
        f,
        indent=2,
        default=str  # REQUIRED for datetime
    )